# Briefing Continuity A/B Test
<!-- Created: 2026-02-28 -->

**A**: Current prompt (no previous briefing context)  
**C**: Same prompt + yesterday's chapter summaries injected  

Goal: Does providing yesterday's topics improve story continuity and reduce redundancy?

In [1]:
import os, time, json, re
from pathlib import Path
from datetime import date, timedelta
from collections import Counter
from dotenv import load_dotenv
from supabase import create_client
from google import genai
from google.genai import types

env_path = Path('../.env.local') if Path('../.env.local').exists() else Path('.env.local')
load_dotenv(env_path)

sb = create_client(os.getenv('NEXT_PUBLIC_SUPABASE_URL'), os.getenv('SUPABASE_SERVICE_ROLE_KEY'))
gemini_client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

print('Connected: Supabase, Gemini')

Connected: Supabase, Gemini


## Fetch Yesterday's Briefing + Extract Chapter Summaries

In [2]:
today = date.today()
yesterday = today - timedelta(days=1)

# Fetch yesterday's EN briefing from DB
prev_briefing = sb.table('wsj_briefings') \
    .select('briefing_text, chapters, date') \
    .eq('category', 'EN') \
    .eq('date', str(yesterday)) \
    .limit(1) \
    .execute()

if not prev_briefing.data:
    # Try 2 days ago (in case yesterday had no briefing)
    day_before = today - timedelta(days=2)
    prev_briefing = sb.table('wsj_briefings') \
        .select('briefing_text, chapters, date') \
        .eq('category', 'EN') \
        .eq('date', str(day_before)) \
        .limit(1) \
        .execute()

if prev_briefing.data:
    prev = prev_briefing.data[0]
    prev_text = prev['briefing_text']
    prev_date = prev['date']
    print(f'Found previous briefing: {prev_date} ({len(prev_text)} chars)')
else:
    prev_text = None
    prev_date = None
    print('No previous briefing found — will skip C variant')

Found previous briefing: 2026-02-27 (9816 chars)


In [3]:
# Extract chapter titles + first sentence from previous briefing
def extract_chapter_summaries(text: str) -> str:
    """Parse [CHAPTER: Title] markers and grab the first sentence after each.
    If markers were already stripped, fall back to paragraph-based extraction."""
    
    chapter_re = re.compile(r'\[CHAPTER:\s*(.+?)\]\s*\n?', re.MULTILINE)
    matches = list(chapter_re.finditer(text))
    
    if matches:
        # Has chapter markers
        summaries = []
        for i, m in enumerate(matches):
            title = m.group(1).strip()
            # Get text between this marker and next (or end)
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            chunk = text[start:end].strip()
            # First sentence
            first_sentence = chunk.split('. ')[0].strip()
            if first_sentence and not first_sentence.endswith('.'):
                first_sentence += '.'
            if title.lower() not in ('opening', 'market snapshot'):
                summaries.append(f'- {title}: {first_sentence}')
        return '\n'.join(summaries)
    
    # Fallback: no chapter markers (already cleaned text)
    # Split into paragraphs, take first sentence of substantial ones
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 100]
    summaries = []
    for p in paragraphs[1:7]:  # Skip opening greeting, take up to 6
        first_sentence = p.split('. ')[0].strip()
        if first_sentence and not first_sentence.endswith('.'):
            first_sentence += '.'
        summaries.append(f'- {first_sentence}')
    return '\n'.join(summaries)

if prev_text:
    yesterday_context = extract_chapter_summaries(prev_text)
    print(f'Extracted context ({len(yesterday_context)} chars, ~{len(yesterday_context)//4} tokens):\n')
    print(yesterday_context)
else:
    yesterday_context = None

Extracted context (564 chars, ~141 tokens):

- Alright, let's jump right in with the biggest story of the day, and it's all about AI.
- Of course, this move by Block is happening against a much bigger, more complicated backdrop of how we should even be using this technology.
- This whole AI race is also creating clear winners and losers in the corporate world.
- This push-pull of AI as both a threat and an opportunity is really playing out in the software sector.
- Speaking of high-stakes competition, the bidding war in the media world is finally over.
- And it’s not the only big deal making headlines.


## Fetch Today's Articles (reuse existing pipeline logic)

In [4]:
# 48h lookback window
cutoff = today - timedelta(hours=48)

items = sb.table('wsj_items') \
    .select('id,title,description,feed_name,published_at,link,briefed') \
    .gte('published_at', cutoff.strftime('%Y-%m-%dT%H:%M:%S')) \
    .order('published_at', desc=True) \
    .execute()

print(f'Articles in window (48h): {len(items.data)}')
cats = Counter(i['feed_name'] for i in items.data)
for cat, count in cats.most_common():
    print(f'  {cat}: {count}')

Articles in window (48h): 148
  BUSINESS_MARKETS: 78
  WORLD: 23
  POLITICS: 18
  ECONOMY: 16
  TECH: 13


In [5]:
# Join with crawl results + LLM analysis
item_ids = [i['id'] for i in items.data]

crawl_map = {}
for i in range(0, len(item_ids), 100):
    batch = item_ids[i:i+100]
    crawls = sb.table('wsj_crawl_results') \
        .select('id,wsj_item_id,content,relevance_flag,relevance_score,llm_same_event,top_image') \
        .in_('wsj_item_id', batch) \
        .eq('relevance_flag', 'ok') \
        .execute()
    for c in crawls.data:
        crawl_map[c['wsj_item_id']] = c

llm_map = {}
crawl_ids = [c['id'] for c in crawl_map.values()]
for i in range(0, len(crawl_ids), 100):
    batch = crawl_ids[i:i+100]
    llms = sb.table('wsj_llm_analysis') \
        .select('crawl_result_id,summary,key_entities,key_numbers,event_type') \
        .in_('crawl_result_id', batch) \
        .execute()
    for l in llms.data:
        llm_map[l['crawl_result_id']] = l

print(f'Crawl results: {len(crawl_map)}')
print(f'LLM analyses: {len(llm_map)}')

Crawl results: 139
LLM analyses: 137


In [6]:
# Assemble articles (simplified — no curation step for test)
RELEVANCE_THRESHOLD = 0.6
articles = []

for item in items.data:
    wid = item['id']
    crawl = crawl_map.get(wid)
    llm = {}
    if crawl and crawl.get('id') in llm_map:
        llm = llm_map[crawl['id']]

    has_quality = False
    if crawl:
        score = crawl.get('relevance_score') or 0
        same_event = crawl.get('llm_same_event', False)
        has_quality = score >= RELEVANCE_THRESHOLD or same_event

    summary = llm.get('summary') or ''
    full_content = (crawl.get('content') or '') if crawl else ''

    base = {
        'title': item['title'],
        'description': item.get('description') or '',
        'category': item['feed_name'],
        'published_at': item.get('published_at', ''),
        'key_entities': llm.get('key_entities', []),
        'key_numbers': [str(n) for n in llm.get('key_numbers', [])],
    }

    if has_quality and summary:
        base['content'] = summary
    elif has_quality and full_content:
        base['content'] = full_content[:800]
    else:
        base['content'] = base['description'] or base['title']

    articles.append(base)

print(f'Assembled {len(articles)} articles')

Assembled 148 articles


## Build Prompts: A (baseline) vs C (with yesterday's context)

In [7]:
# Import the system prompt from the production script
import sys
sys.path.insert(0, '../scripts')
from importlib import import_module
gen = import_module('8_generate_briefing')
SYSTEM_PROMPT = gen.BRIEFING_SYSTEM_EN

# Format articles text
def format_article(a):
    parts = [f"[{a['category']}] {a['title']}"]
    if a.get('published_at'):
        parts.append(f"  Published: {a['published_at'][:16].replace('T', ' ')}")
    if a.get('description'): parts.append(f"  Desc: {a['description']}")
    if a.get('content'): parts.append(f"  Content: {a['content']}")
    if a.get('key_entities'): parts.append(f"  Entities: {', '.join(a['key_entities'])}")
    if a.get('key_numbers'): parts.append(f"  Numbers: {', '.join(a['key_numbers'])}")
    return '\n'.join(parts)

today_str = today.strftime('%A, %B %d, %Y')
articles_text = f"Date: {today_str}\nToday's articles ({len(articles)} total):\n\n"
articles_text += '\n\n'.join(format_article(a) for a in articles)

# --- Prompt A: baseline (no context) ---
prompt_a = SYSTEM_PROMPT + '\n\n' + articles_text

# --- Prompt C: with yesterday's chapter summaries ---
if yesterday_context:
    context_block = (
        f"\n\nYesterday's briefing ({prev_date}) covered these topics:\n"
        f"{yesterday_context}\n\n"
        f"Use this for continuity — connect follow-up stories naturally "
        f"(e.g., 'As we discussed yesterday...'). "
        f"Avoid repeating the same coverage depth for stories already covered in detail. "
        f"Focus your depth on NEW developments today."
    )
    prompt_c = SYSTEM_PROMPT + context_block + '\n\n' + articles_text
else:
    prompt_c = None

print(f'Prompt A: {len(prompt_a)//4:,} tokens (est)')
if prompt_c:
    print(f'Prompt C: {len(prompt_c)//4:,} tokens (est)')
    print(f'Context overhead: +{(len(prompt_c) - len(prompt_a))//4:,} tokens')
else:
    print('Prompt C: skipped (no previous briefing)')

Prompt A: 48,440 tokens (est)
Prompt C: 48,652 tokens (est)
Context overhead: +212 tokens


## Generate A vs C

In [8]:
THINKING_BUDGET = 4096
MODEL = 'gemini-2.5-pro'

def generate(prompt: str, label: str) -> dict:
    print(f'Generating [{label}]...', end=' ', flush=True)
    start = time.time()
    resp = gemini_client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=8192,
            temperature=0.6,
            thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET),
        ),
    )
    elapsed = time.time() - start
    usage = resp.usage_metadata
    thinking = getattr(usage, 'thoughts_token_count', None) or getattr(usage, 'thinking_token_count', 0) or 0
    
    result = {
        'text': resp.text,
        'elapsed': elapsed,
        'input_tokens': usage.prompt_token_count or 0,
        'output_tokens': usage.candidates_token_count or 0,
        'thinking_tokens': thinking,
        'words': len(resp.text.split()),
    }
    print(f'{result["words"]} words, {elapsed:.1f}s (in:{result["input_tokens"]:,} out:{result["output_tokens"]:,} think:{thinking:,})')
    return result

result_a = generate(prompt_a, 'A: baseline')
if prompt_c:
    result_c = generate(prompt_c, 'C: with context')
else:
    result_c = None
    print('Skipping C — no previous briefing context available')

Generating [A: baseline]... 1576 words, 44.3s (in:48,199 out:2,073 think:2,900)
Generating [C: with context]... 1708 words, 41.2s (in:48,396 out:2,228 think:2,365)


## Compare Results

In [9]:
# Cost calculation
COST_PER_1M = {'input': 1.25, 'output': 10.0, 'thinking': 5.0}

def calc_cost(r):
    return (
        r['input_tokens'] * COST_PER_1M['input'] / 1e6 +
        r['output_tokens'] * COST_PER_1M['output'] / 1e6 +
        r['thinking_tokens'] * COST_PER_1M['thinking'] / 1e6
    )

print('=== COMPARISON ===')
print(f'{"":>20} {"A (baseline)":>15} {"C (context)":>15}')
print(f'{"Words":>20} {result_a["words"]:>15,}', end='')
if result_c: print(f' {result_c["words"]:>15,}')
else: print(' —')
print(f'{"Time (s)":>20} {result_a["elapsed"]:>15.1f}', end='')
if result_c: print(f' {result_c["elapsed"]:>15.1f}')
else: print(' —')
print(f'{"Input tokens":>20} {result_a["input_tokens"]:>15,}', end='')
if result_c: print(f' {result_c["input_tokens"]:>15,}')
else: print(' —')
print(f'{"Output tokens":>20} {result_a["output_tokens"]:>15,}', end='')
if result_c: print(f' {result_c["output_tokens"]:>15,}')
else: print(' —')
print(f'{"Cost ($)":>20} {calc_cost(result_a):>15.4f}', end='')
if result_c: print(f' {calc_cost(result_c):>15.4f}')
else: print(' —')

if result_c:
    extra_cost = calc_cost(result_c) - calc_cost(result_a)
    print(f'\nContext overhead cost: ${extra_cost:.4f} per briefing (${extra_cost*30:.2f}/month)')

=== COMPARISON ===
                        A (baseline)     C (context)
               Words           1,576           1,708
            Time (s)            44.3            41.2
        Input tokens          48,199          48,396
       Output tokens           2,073           2,228
            Cost ($)          0.0955          0.0946

Context overhead cost: $-0.0009 per briefing ($-0.03/month)


In [10]:
# Print both briefings for manual comparison
print('=' * 80)
print('BRIEFING A (baseline)')
print('=' * 80)
print(result_a['text'][:3000])
print('\n... [truncated] ...\n')

if result_c:
    print('=' * 80)
    print('BRIEFING C (with yesterday context)')
    print('=' * 80)
    print(result_c['text'][:3000])
    print('\n... [truncated] ...')

BRIEFING A (baseline)
[CHAPTER: Opening]
Good morning, and welcome to the show. It is Saturday, February 28, 2026. It’s been a wild week, and we’ve got a lot to unpack. Today, we’re looking at a major showdown between the White House and Silicon Valley’s biggest AI labs that has huge implications for national security. We’ll also dig into why the markets are suddenly terrified of that same AI, and what it means for your portfolio. And later, we’ll check in on a nasty Hollywood takeover battle that finally has a winner.

[CHAPTER: AI's Washington Standoff]
Let's start in Washington, where a fascinating and frankly, pretty tense, drama is playing out. The Trump administration has officially ordered all U.S. agencies to stop using artificial intelligence from the startup Anthropic. This is a huge escalation in a dispute that’s been simmering between the company and the Pentagon.

So what's this all about? It boils down to guardrails. The Pentagon, led by Defense Secretary Pete Hegseth, ha

In [11]:
# Save full outputs for detailed review
output_dir = Path('../notebooks/tts_outputs/text')
output_dir.mkdir(parents=True, exist_ok=True)

today_file = today.strftime('%Y-%m-%d')

with open(output_dir / f'ab-test-A-{today_file}.txt', 'w') as f:
    f.write(result_a['text'])
print(f'Saved: ab-test-A-{today_file}.txt')

if result_c:
    with open(output_dir / f'ab-test-C-{today_file}.txt', 'w') as f:
        f.write(result_c['text'])
    print(f'Saved: ab-test-C-{today_file}.txt')

# Save test metadata
meta = {
    'date': str(today),
    'prev_briefing_date': prev_date,
    'article_count': len(articles),
    'yesterday_context_chars': len(yesterday_context) if yesterday_context else 0,
    'A': {k: v for k, v in result_a.items() if k != 'text'},
}
if result_c:
    meta['C'] = {k: v for k, v in result_c.items() if k != 'text'}

with open(output_dir / f'ab-test-meta-{today_file}.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Saved: ab-test-meta-{today_file}.json')

Saved: ab-test-A-2026-02-28.txt
Saved: ab-test-C-2026-02-28.txt
Saved: ab-test-meta-2026-02-28.json
